# 21 — Deterministic checks first, Ragas as a second signal
Ragas judges factual overlap with a local Ollama model. It cannot prove that
an approval gate or a customer scope was enforced.



**Advanced follow-up to lesson 12 (evaluation).**


## Problem

Contract tests cannot measure whether a recommendation is factually close to a reference.


## Concept

Ragas adds a model-judged score alongside deterministic safety gates.

Ragas FactualCorrectness calls an evaluator model to compare a response with a reference.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly   ◀ YOU ARE HERE
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
report → deterministic PASS/FAIL; response + reference → Ragas/Ollama score
```

reference + response → local Ollama evaluator → factual score. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Run a deterministic report check

Run a deterministic report check.


In [ ]:
import asyncio
import os
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.control import validate_report
from workshop.ragas_eval import factual_score, load_cases
from workshop.schemas import DecisionReport, Evidence

report = DecisionReport(
    recommendation="pilot",
    rationale="A limited pilot is justified while deletion guarantees remain unverified.",
    evidence=[Evidence(source_id="architecture.md", claim="Private VPC deployment is possible."),
              Evidence(source_id="security.md", claim="Deletion guarantees are unverified.")],
)
print("Deterministic report check:", validate_report(report))
cases = load_cases()
print("Versioned evaluation cases:", [case["id"] for case in cases])



## Run a local model-judged score only when requested
Scores may vary by model and run; record the model and input dataset.



## Step 2 — Compare response and reference with the local Ragas evaluator

Compare response and reference with the local Ragas evaluator.


In [ ]:
if os.getenv("WORKSHOP_RUN_LIVE_DEMOS") == "true":
    for case in cases:
        print(case["id"], asyncio.run(factual_score(case["response"], case["reference"])))
else:
    print("Set WORKSHOP_RUN_LIVE_DEMOS=true for the Ragas/Ollama metric.")


## Failure mode to know about

A high factual score cannot catch a missing approval or cross-customer memory leak.


## Production implication

Version the evaluation set and compare scores across model changes; never use one score alone.


## Your turn

Change the reference and compare the score while keeping safety tests fixed.

**Checkpoint:** The score is reported separately from validation and approval results.
